## 0. Setup

In [1]:
import pandas as pd
import numpy as np
import os, glob
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

path = os.path.join('..', 'data', 'Track_D_Mental_Health')

T16    = 'Have you ever sought treatment for a mental health issue from a mental health professional?'
T_     = 'Have you ever sought treatment for a mental health disorder from a mental health professional?'
SEEK_A = 'Does your employer offer resources to learn more about mental health concerns and options for seeking help?'
SEEK_B = 'Does your employer offer resources to learn more about mental health disorders and options for seeking help?'
REMOTE = 'Do you work remotely?'
EMP    = 'How many employees does your company or organization have?'

dfs = []
for f in sorted(glob.glob(os.path.join(path, '*.csv'))):
    yr    = int(os.path.basename(f).replace('Mental_Health_Survey_', '').replace('.csv', ''))
    frame = pd.read_csv(f, low_memory=False)
    frame['year'] = yr
    for c in [T16, T_]:
        if c in frame.columns:
            frame = frame.rename(columns={c: 'treatment'})
            break
    for c in list(frame.columns):
        if 'interferes with your work when being treated effectively' in c:
            frame = frame.rename(columns={c: 'work_interfere'})
            break
    for c in [SEEK_A, SEEK_B]:
        if c in frame.columns:
            frame = frame.rename(columns={c: 'seek_help'})
            break
    if REMOTE in frame.columns:
        frame = frame.rename(columns={REMOTE: 'remote_work'})
    if EMP in frame.columns:
        frame = frame.rename(columns={EMP: 'no_employees'})
    dfs.append(frame)

df = pd.concat(dfs, ignore_index=True)

# Yes/No only in 2016; later years already store integers
if df['treatment'].dtype == object:
    df['treatment'] = df['treatment'].map({'Yes': 1, 'No': 0}).fillna(df['treatment'])
df['treatment'] = pd.to_numeric(df['treatment'], errors='coerce')
df = df.dropna(subset=['treatment'])
df['treatment'] = df['treatment'].astype(int)

# --- Feature selection ---
X = df.drop(columns=['treatment', 'year']).copy()

# Drop columns with >50% missing
miss_rate = X.isnull().mean()
X = X[miss_rate[miss_rate <= 0.5].index]

# Drop high-cardinality free-text columns (gender 164 uniques, why-or-why-not 2500+)
HIGH_CARD = 50
X = X[[c for c in X.columns if X[c].nunique() <= HIGH_CARD]]

# --- Separate categoricals vs numericals ---
cat_cols = [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]
num_cols = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]

# --- Fill NaN ---
for c in cat_cols:
    X[c] = X[c].fillna('Unknown')
for c in num_cols:
    X[c] = X[c].fillna(X[c].median())

# --- Label encode categoricals ---
encoders = {}
for c in cat_cols:
    uniq    = sorted(X[c].unique(), key=str)
    mapping = {val: idx for idx, val in enumerate(uniq)}
    encoders[c] = mapping
    X[c] = X[c].map(mapping)

# --- Normalize ALL features (label-encoded cats are now ints, need scaling too) ---
for c in X.columns:
    X[c] = X[c].astype(float)
    mu, sigma = X[c].mean(), X[c].std()
    X[c] = (X[c] - mu) / (sigma + 1e-8)

# --- Convert to numpy, split ---
X_np = X.values.astype(np.float64)
y_np = df.loc[X.index, 'treatment'].values.astype(np.float64)

X_train, X_test, y_train, y_test = train_test_split(X_np, y_np, test_size=0.2, random_state=42)

print(f'Features: {X.shape[1]} ({len(cat_cols)} categorical + {len(num_cols)} numerical)')
print(f'X_train: {X_train.shape}, X_test: {X_test.shape}')
print(f'y_train: {y_train.shape}, y_test: {y_test.shape}')
print(f'Class balance: {y_train.mean():.2%} positive')


Features: 18 (17 categorical + 1 numerical)
X_train: (2746, 18), X_test: (687, 18)
y_train: (2746,), y_test: (687,)
Class balance: 58.05% positive


## 1. K-Means Clustering (from scratch)

In [2]:
K          = 2
MAX_ITER_KM = 100

rng     = np.random.RandomState(42)
indices = rng.choice(len(X_train), K, replace=False)
centroids = X_train[indices].copy()

for it in range(MAX_ITER_KM):
    dists  = np.linalg.norm(X_train[:, None] - centroids[None, :], axis=2)
    labels = np.argmin(dists, axis=1)

    new_centroids = np.array([X_train[labels == k].mean(axis=0) for k in range(K)])
    if np.allclose(centroids, new_centroids):
        break
    centroids = new_centroids

km_iters     = it + 1
km_labels    = labels
km_centroids = centroids

print(f'K-Means converged in {km_iters} iterations')
print(f'Centroid shape: {km_centroids.shape}')

# Purity: majority label per cluster / total
majority_sum = 0
for k in range(K):
    mask = km_labels == k
    if mask.sum() == 0:
        continue
    majority_sum += np.bincount(y_train[mask].astype(int)).max()

km_purity = majority_sum / len(y_train)
print(f'K-Means purity: {km_purity:.4f}')


K-Means converged in 28 iterations
Centroid shape: (2, 18)
K-Means purity: 0.5805


## 2. K-Medoid Clustering (from scratch)

In [3]:
K          = 2
MAX_ITER_KD = 100

rng        = np.random.RandomState(42)
medoid_idx = rng.choice(len(X_train), K, replace=False)

for it in range(MAX_ITER_KD):
    medoids = X_train[medoid_idx]
    dists   = np.linalg.norm(X_train[:, None] - medoids[None, :], axis=2)
    labels  = np.argmin(dists, axis=1)

    new_medoid_idx = medoid_idx.copy()
    for k in range(K):
        members = np.where(labels == k)[0]
        if len(members) == 0:
            continue
        # Find the point that minimizes total intra-cluster distance
        intra = np.linalg.norm(
            X_train[members][:, None] - X_train[members][None, :], axis=2
        )
        new_medoid_idx[k] = members[np.argmin(intra.sum(axis=1))]

    if np.array_equal(medoid_idx, new_medoid_idx):
        break
    medoid_idx = new_medoid_idx

kd_iters  = it + 1
kd_labels = labels

print(f'K-Medoid converged in {kd_iters} iterations')

# Purity
majority_sum = 0
for k in range(K):
    mask = kd_labels == k
    if mask.sum() == 0:
        continue
    majority_sum += np.bincount(y_train[mask].astype(int)).max()
kd_purity = majority_sum / len(y_train)
print(f'K-Medoid purity: {kd_purity:.4f}')

# Within-cluster sum of distances comparison
def wcsd(X, labels, centers):
    return sum(
        np.linalg.norm(X[labels == k] - centers[k], axis=1).sum()
        for k in range(len(centers))
    )

km_wcsd = wcsd(X_train, km_labels, km_centroids)
kd_wcsd = wcsd(X_train, kd_labels, X_train[medoid_idx])
print(f'K-Means WCSD:  {km_wcsd:.2f}')
print(f'K-Medoid WCSD: {kd_wcsd:.2f}')
print(f'Lower: {"K-Means" if km_wcsd < kd_wcsd else "K-Medoid"}')


K-Medoid converged in 2 iterations
K-Medoid purity: 0.5805
K-Means WCSD:  10941.07
K-Medoid WCSD: 12554.08
Lower: K-Means


## 3. Perceptron (from scratch)

In [4]:
LR_P     = 0.1
EPOCHS_P = 50

w    = np.zeros(X_train.shape[1])
bias = 0.0
train_accs_p = []

for epoch in range(EPOCHS_P):
    for i in range(len(X_train)):
        y_hat = 1 if np.dot(w, X_train[i]) + bias > 0 else 0
        err   = y_train[i] - y_hat
        w    += LR_P * err * X_train[i]
        bias += LR_P * err

    preds = (X_train @ w + bias > 0).astype(int)
    train_accs_p.append((preds == y_train).mean())

# Test accuracy
preds_test_p  = (X_test @ w + bias > 0).astype(int)
perc_test_acc = (preds_test_p == y_test).mean()
print(f'Perceptron test accuracy: {perc_test_acc:.4f}')

plt.figure(figsize=(8, 4))
plt.plot(range(1, EPOCHS_P + 1), train_accs_p)
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Perceptron Training Accuracy')
plt.tight_layout()
plt.show()


Perceptron test accuracy: 0.6667


C:\Users\zawia\AppData\Local\Temp\ipykernel_4968\3038913819.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Delta Rule — Batch Gradient Descent (from scratch)

In [5]:
LR_D     = 0.01
EPOCHS_D = 100

np.random.seed(42)
w_d    = np.random.randn(X_train.shape[1]) * 0.01
bias_d = 0.0
mse_list = []

for epoch in range(EPOCHS_D):
    y_hat  = X_train @ w_d + bias_d
    err    = y_train - y_hat
    n      = len(y_train)
    grad_w = -(2 / n) * (X_train.T @ err)
    grad_b = -(2 / n) * err.sum()
    w_d   -= LR_D * grad_w
    bias_d -= LR_D * grad_b
    mse_list.append((err ** 2).mean())

# Test accuracy (threshold at 0.5)
y_hat_test_d   = X_test @ w_d + bias_d
preds_test_d   = (y_hat_test_d > 0.5).astype(int)
delta_test_acc = (preds_test_d == y_test).mean()
print(f'Delta Rule final MSE: {mse_list[-1]:.4f}')
print(f'Delta Rule test accuracy: {delta_test_acc:.4f}')

plt.figure(figsize=(8, 4))
plt.plot(range(1, EPOCHS_D + 1), mse_list)
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.title('Delta Rule MSE')
plt.tight_layout()
plt.show()

print(f'Perceptron ran {EPOCHS_P} epochs with step updates; '
      f'Delta Rule ran {EPOCHS_D} epochs with continuous gradient signal.')


Delta Rule final MSE: 0.1462
Delta Rule test accuracy: 0.8413
Perceptron ran 50 epochs with step updates; Delta Rule ran 100 epochs with continuous gradient signal.


C:\Users\zawia\AppData\Local\Temp\ipykernel_4968\2177150126.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. MLP with Backpropagation (from scratch)

In [6]:
LR_MLP     = 0.01
EPOCHS_MLP = 200
H1, H2     = 16, 8

def relu(x):    return np.maximum(0, x)
def relu_d(x):  return (x > 0).astype(float)
def sigmoid(x): return 1 / (1 + np.exp(-np.clip(x, -500, 500)))
def bce(y, p):  return -np.mean(y * np.log(p + 1e-8) + (1 - y) * np.log(1 - p + 1e-8))

n_feat = X_train.shape[1]
np.random.seed(42)
# He initialization: std = sqrt(2 / fan_in) -- prevents dead ReLU at start
W1 = np.random.randn(n_feat, H1) * np.sqrt(2 / n_feat);  b1 = np.zeros(H1)
W2 = np.random.randn(H1, H2)     * np.sqrt(2 / H1);      b2 = np.zeros(H2)
W3 = np.random.randn(H2, 1)      * np.sqrt(2 / H2);      b3 = np.zeros(1)

losses, accs_mlp = [], []
n = len(X_train)

for epoch in range(EPOCHS_MLP):
    # Forward
    z1 = X_train @ W1 + b1;  a1 = relu(z1)
    z2 = a1 @ W2 + b2;       a2 = relu(z2)
    z3 = a2 @ W3 + b3;       a3 = sigmoid(z3).flatten()

    loss = bce(y_train, a3)
    acc  = ((a3 > 0.5).astype(int) == y_train).mean()
    losses.append(loss)
    accs_mlp.append(acc)

    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1:3d}: loss={loss:.4f}, acc={acc:.4f}')

    # Backward
    dL_da3 = -(y_train / (a3 + 1e-8)) + (1 - y_train) / (1 - a3 + 1e-8)
    dL_dz3 = dL_da3 * a3 * (1 - a3)
    dW3    = a2.T @ dL_dz3.reshape(-1, 1) / n
    db3    = dL_dz3.mean(axis=0, keepdims=True)

    dL_da2 = dL_dz3.reshape(-1, 1) @ W3.T
    dL_dz2 = dL_da2 * relu_d(z2)
    dW2    = a1.T @ dL_dz2 / n
    db2    = dL_dz2.mean(axis=0)

    dL_da1 = dL_dz2 @ W2.T
    dL_dz1 = dL_da1 * relu_d(z1)
    dW1    = X_train.T @ dL_dz1 / n
    db1    = dL_dz1.mean(axis=0)

    W3 -= LR_MLP * dW3;  b3 -= LR_MLP * db3
    W2 -= LR_MLP * dW2;  b2 -= LR_MLP * db2
    W1 -= LR_MLP * dW1;  b1 -= LR_MLP * db1

# Test accuracy
z1t = X_test @ W1 + b1;   a1t = relu(z1t)
z2t = a1t @ W2 + b2;      a2t = relu(z2t)
z3t = a2t @ W3 + b3;      a3t = sigmoid(z3t).flatten()
mlp_test_acc = ((a3t > 0.5).astype(int) == y_test).mean()
print(f'\nMLP test accuracy: {mlp_test_acc:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(range(1, len(losses) + 1), losses)
axes[0].set_xlabel('Epoch');    axes[0].set_ylabel('Loss');     axes[0].set_title('MLP Loss')
axes[1].plot(range(1, len(accs_mlp) + 1), accs_mlp)
axes[1].set_xlabel('Epoch');    axes[1].set_ylabel('Accuracy'); axes[1].set_title('MLP Training Accuracy')
plt.tight_layout()
plt.show()


Epoch  10: loss=0.7975, acc=0.5849
Epoch  20: loss=0.7453, acc=0.5870
Epoch  30: loss=0.7082, acc=0.5918
Epoch  40: loss=0.6816, acc=0.5980
Epoch  50: loss=0.6622, acc=0.6042
Epoch  60: loss=0.6477, acc=0.6096
Epoch  70: loss=0.6367, acc=0.6158
Epoch  80: loss=0.6280, acc=0.6220
Epoch  90: loss=0.6209, acc=0.6304
Epoch 100: loss=0.6149, acc=0.6377
Epoch 110: loss=0.6098, acc=0.6493
Epoch 120: loss=0.6052, acc=0.6602
Epoch 130: loss=0.6010, acc=0.6682
Epoch 140: loss=0.5971, acc=0.6784
Epoch 150: loss=0.5935, acc=0.6857
Epoch 160: loss=0.5900, acc=0.6890


Epoch 170: loss=0.5867, acc=0.6930
Epoch 180: loss=0.5836, acc=0.6981
Epoch 190: loss=0.5805, acc=0.7017
Epoch 200: loss=0.5776, acc=0.7054

MLP test accuracy: 0.7132


C:\Users\zawia\AppData\Local\Temp\ipykernel_4968\3243825365.py:67: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Model Comparison

In [7]:
import pandas as pd

cmp = pd.DataFrame({
    'Model':         ['K-Means', 'K-Medoid', 'Perceptron', 'Delta Rule', 'MLP'],
    'Test Accuracy':  [f'{km_purity:.4f}', f'{kd_purity:.4f}',
                       f'{perc_test_acc:.4f}', f'{delta_test_acc:.4f}',
                       f'{mlp_test_acc:.4f}'],
    'Notes':          ['unsupervised - purity not accuracy',
                       'unsupervised - purity not accuracy',
                       'linear, step activation',
                       'linear, continuous output',
                       '2 hidden layers, backprop'],
})
print(cmp.to_string(index=False))


     Model Test Accuracy                              Notes
   K-Means        0.5805 unsupervised - purity not accuracy
  K-Medoid        0.5805 unsupervised - purity not accuracy
Perceptron        0.6667            linear, step activation
Delta Rule        0.8413          linear, continuous output
       MLP        0.7132          2 hidden layers, backprop


## Phase 5 Reflection

**1. What were the final test accuracies for Perceptron, Delta Rule, and MLP?
   Which performed best and why?**

Perceptron: 66.67%, Delta Rule: 84.13%, MLP: 71.32%.
Delta Rule performed best despite being a linear model. It uses continuous gradient
descent on MSE with batch updates, which converges more reliably on this low-signal
dataset of 18 ordinal features. The Perceptron's binary weight updates are too coarse
for a dataset where the decision boundary is not cleanly linearly separable. The MLP
outperformed the Perceptron by learning non-linear combinations of features, but was
capped by 200 epochs; with more training it would likely exceed Delta Rule accuracy.

**2. How many iterations did K-Means and K-Medoid take to converge?
   Which had lower within-cluster sum of distances?**

K-Means converged in 28 iterations; K-Medoid converged in 2 iterations.
K-Means WCSD: 10941.07 vs K-Medoid WCSD: 12554.08 — K-Means is lower.
K-Means achieves tighter clusters because centroids are means that minimize total
squared distance by definition. K-Medoid converges faster (medoids jump directly
to an existing data point) but cannot match the geometric optimality of the centroid.

**3. Did the MLP loss consistently decrease over 200 epochs?
   What does the curve shape tell you about the learning rate?**

Yes — the MLP loss decreased consistently from 0.7975 (epoch 10) to 0.5776 (epoch 200)
with no oscillation. The smooth, steady descent indicates that LR=0.01 is appropriate
for this architecture and dataset: large enough to make consistent progress, small enough
not to overshoot. The slow early phase (epochs 1–50) reflects the time needed for
He-initialized weights to align to the data manifold.

**4. Why does K-Means purity not equal classification accuracy,
   and what does it measure instead?**

Both clusters in this run have purity = 0.5805, which equals the majority class
proportion. This reveals that K-Means found no discriminative cluster structure — both
clusters are dominated by the same majority class (treated=1 at 58%). Purity measures
intra-cluster label homogeneity: for each cluster, it counts the fraction belonging to
the most common label. It does not penalize trivially assigning all points to one cluster.
Classification accuracy, by contrast, requires matching each point to the correct label,
penalizing both false positives and false negatives symmetrically.
